# Anchoring of merger offer prices: analysis

Takes the enriched SDC deal file and CRSP acquirer returns (built in `01_wrds_data_build.ipynb`), forms AnchorDistance deciles, computes CAR and BHAR, and produces the report tables in `results/`. Run the cells top to bottom.

**Inputs** (not in the repository; see the *Data sources* section of the README): the enriched SDC file, the SDC pull with `pctown`, the acquirer daily-return files, the Fama-French daily factors.

**Definitions**
- AnchorDistance = offer price / target 52-week high - 1. Decile 1 = lowest (paid most below the high), decile 10 = highest.
- Working sample for the deciles: matched deals where target and acquirer are different companies **and** the acquirer ends with 100% of the target (`pctown == 100`).
- CAR(-5,+5): market model fitted on t-252..t-31 trading days; t = 0 is the first trading day on or after the announcement.
- BHAR: acquirer minus market buy-and-hold return over four consecutive 63-trading-day quarters after t = 0, and the full 252 days.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 250, "display.max_columns", 30)

# ---- file locations (relative to this notebook) ----
ROOT = Path("..")
SDC_FILE = ROOT / "data/interim/sdc_pr_52w_high_decile.csv"        # enriched SDC file
PCT_FILE = ROOT / "data/raw/sdc_us_public_2005_2025_pct.csv"       # SDC pull that carries pctown
FF_DAILY = ROOT / "data/external/F-F_Research_Data_5_Factors_2x3_daily.csv"
ACQ_FILE = lambda k: ROOT / f"data/interim/acquirer_decile_{k}.csv"  # acquirer daily returns, files 1..10
RESULTS = ROOT / "results"
PROCESSED = ROOT / "data/processed"

# ---- design parameters ----
KEYS = ["master_cusip", "acusip", "dateann"]      # one deal = target CUSIP + acquirer CUSIP + announcement date
SDC_ORIGINAL = [                                   # SDC columns from the WRDS query; never modified here
    "master_cusip", "dateann", "amv", "dateannorig", "status", "acusip", "aup", "tup",
    "anation", "tnation", "tpublic", "apublic", "tticker", "deal_value", "pr", "consid", "considoff",
]
TEXT_COLS = ["master_cusip", "acusip", "dateann", "tcusip8", "acusip8", "tcusip6", "acusip6",
             "tup", "aup", "tup6", "aup6"]         # read as text so leading zeros survive
PERMNO_COLS = ["tpermno", "apermno", "tpermno_final", "apermno_final", "tpermno.1"]  # written back as integers

EST_START, EST_END = -252, -31                     # market-model estimation window (trading days)
EVT_START, EVT_END = -5, 5                         # event window
MIN_EST_OBS = 100                                  # minimum non-missing returns in the estimation window
QUARTERS = [(1, 63), (64, 126), (127, 189), (190, 252)]
HORIZON = 252
N_BOOT, N_PERM, SEED = 5000, 10000, 0

PERIODS = {"car": "CAR[-5,+5]", "q1": "BHAR Q1", "q2": "BHAR Q2", "q3": "BHAR Q3", "q4": "BHAR Q4", "bhar12": "BHAR 12m"}

## 1. Loaders
SDC file, Fama-French market return (also the trading calendar), and acquirer daily returns.

In [2]:
def load_ff(path, skiprows, fmt):
    """Read a Kenneth French CSV into a numeric frame indexed by date.

    French CSVs have a text description, then a header row, then data, then a
    copyright footer. Dates are parsed with errors="coerce" so the footer and any
    annual rows become missing, and are then dropped.
    """
    df = pd.read_csv(path, skiprows=skiprows, index_col=0)
    df.index = pd.to_datetime(df.index, format=fmt, errors="coerce")
    df = df[df.index.notna()]
    return df.apply(pd.to_numeric, errors="coerce")

def build_market_return(ff):
    """Return the market return in decimal, (Mkt-RF + RF) / 100, in column mkt_ret.

    French files report percent. Values above 50% in absolute size are set to
    missing (these are the -99.99 / -999 missing-data codes).
    """
    out = pd.DataFrame(index=ff.index)
    out["mkt_ret"] = (ff["Mkt-RF"] + ff["RF"]) / 100.0
    out.loc[out["mkt_ret"].abs() > 0.5, "mkt_ret"] = pd.NA
    return out

def load_market(path):
    """Daily market return (decimal) as a float Series indexed by trading date.

    The index doubles as the trading-day calendar for all event windows.
    """
    ff = load_ff(path, skiprows=3, fmt="%Y%m%d")
    return build_market_return(ff)["mkt_ret"].astype(float).dropna()

def load_sdc(path):
    """Read the enriched SDC file with identifier columns as text."""
    return pd.read_csv(path, dtype={c: str for c in TEXT_COLS}, low_memory=False)

def read_acquirer_returns():
    """Read acquirer_decile_1..10.csv into one table of daily returns.

    Only the deal key, PERMNO, date and return are kept. The decile stored in
    these files is ignored: deciles always come from the SDC file, so the
    return files never need to be rewritten.
    """
    parts = [
        pd.read_csv(
            ACQ_FILE(k),
            usecols=["permno", "dlycaldt", "dlyret"] + KEYS,
            dtype={"master_cusip": str, "acusip": str, "dateann": str},
        )
        for k in range(1, 11)
    ]
    acq = pd.concat(parts, ignore_index=True)
    acq["dlycaldt"] = pd.to_datetime(acq["dlycaldt"])
    acq["dlyret"] = pd.to_numeric(acq["dlyret"], errors="coerce")
    return acq

def attach_deciles(acq, sdc):
    """Attach each deal's anchor_decile (from the SDC file) to its acquirer return rows.

    Uses the first copy of each deal key and keeps only deals that have a decile.
    """
    dedup = sdc.drop_duplicates(subset=KEYS)
    keys = dedup.loc[dedup["anchor_decile"].notna(), KEYS + ["anchor_decile"]]
    out = acq.merge(keys, on=KEYS, how="inner")
    out["anchor_decile"] = out["anchor_decile"].astype(int)
    return out

## 2. Working sample and deciles
`build_sample` adds `pctown`, `include_decile` and `anchor_decile` and asserts that the original SDC columns are unchanged. Re-running is safe: derived columns are overwritten.

In [3]:
def add_pctown(sdc, pct_path):
    """Add pctown (percent owned after the deal) from the pct file, matched on the deal keys.

    Repeated keys take the first copy. If pctown is already in the frame it is left as is.
    Returns a new frame; original columns are untouched.
    """
    if "pctown" in sdc.columns:
        return sdc.copy()
    pct = pd.read_csv(
        pct_path,
        dtype={"master_cusip": str, "acusip": str, "dateann": str},
        usecols=KEYS + ["pctown"],
        low_memory=False,
    )
    pct = pct.drop_duplicates(subset=KEYS)
    return sdc.merge(pct, on=KEYS, how="left", validate="many_to_one")

def add_include_decile(sdc):
    """Set include_decile: 'y' only for third-party deals with a 100% final stake.

    A deal is 'y' when target and acquirer PERMNO differ (tpermno_final != apermno_final)
    AND pctown == 100. Everything else is 'n', including deals with missing pctown.
    Derived column, overwritten on each run. Returns a new frame.
    """
    out = sdc.copy()
    different_company = out["tpermno_final"] != out["apermno_final"]
    full_stake = out["pctown"] == 100
    out["include_decile"] = np.where(different_company & full_stake, "y", "n")
    return out

def add_anchor_decile(sdc):
    """Set anchor_decile: AnchorDistance decile within matched deals with include_decile == 'y'.

    Decile 1 is the lowest AnchorDistance, decile 10 the highest. Other rows get NaN.
    Derived column, overwritten on each run. Returns a new frame.
    """
    out = sdc.copy()
    out["anchor_decile"] = float("nan")
    mask = (
        (out["is_matched"] == "y")
        & (out["include_decile"] == "y")
        & out["anchor_distance"].notna()
    )
    out.loc[mask, "anchor_decile"] = pd.qcut(out.loc[mask, "anchor_distance"], 10, labels=False) + 1
    return out

def build_sample(sdc, pct_path):
    """Add pctown, include_decile and anchor_decile, and check the original SDC columns are unchanged."""
    original = sdc.copy()
    out = add_anchor_decile(add_include_decile(add_pctown(sdc, pct_path)))
    assert len(out) == len(original), "row count changed"
    assert out[SDC_ORIGINAL].equals(original[SDC_ORIGINAL]), "original SDC columns changed"
    return out

def sample_summary(sdc):
    """Return distinct-deal counts that describe the working sample (one row per deal key)."""
    d = sdc.drop_duplicates(subset=KEYS)
    return {
        "deals": len(d),
        "pctown_missing": int(d["pctown"].isna().sum()),
        "pctown_100": int((d["pctown"] == 100).sum()),
        "include_decile_y": int((d["include_decile"] == "y").sum()),
        "deals_with_decile": int(d["anchor_decile"].notna().sum()),
    }

def save_sdc(sdc, path):
    """Write the enriched SDC file back to disk, PERMNO columns as integers (text format unchanged)."""
    out = sdc.copy()
    for c in PERMNO_COLS:
        out[c] = out[c].astype("Int64")
    out.to_csv(path, index=False)

In [4]:
sdc = build_sample(load_sdc(SDC_FILE), PCT_FILE)
sample_summary(sdc)

{'deals': 5944,
 'pctown_missing': 724,
 'pctown_100': 2770,
 'include_decile_y': 2747,
 'deals_with_decile': 1461}

In [5]:
dedup = sdc.drop_duplicates(subset=KEYS)
dedup["anchor_decile"].value_counts().sort_index().rename("deals").to_frame()

,deals
anchor_decile,
1.0,147
2.0,146
3.0,146
4.0,146
5.0,146
6.0,146
7.0,146
8.0,146
9.0,146


In [6]:
# Only needed the first time (writes pctown / include_decile / anchor_decile into the SDC file):
# save_sdc(sdc, SDC_FILE)

## 3. Event-study returns
CAR(-5,+5) from a market model, and quarterly / 12-month BHAR. t = 0 is the first trading day on or after the announcement; the trading calendar is the index of the market-return series.

In [7]:
def event_day_position(calendar, announce_date):
    """Position of t = 0 in the trading calendar: first trading day on or after the announcement (None past the end)."""
    i0 = int(calendar.searchsorted(pd.Timestamp(announce_date), side="left"))
    return i0 if i0 < len(calendar) else None

def fit_market_model(y, x):
    """OLS of y on x with an intercept, using only days where both are non-missing.

    Returns (alpha, beta, n_obs).
    """
    ok = ~(np.isnan(y) | np.isnan(x))
    n = int(ok.sum())
    if n < 3:
        return np.nan, np.nan, n
    X = np.column_stack([np.ones(n), x[ok]])
    alpha, beta = np.linalg.lstsq(X, y[ok], rcond=None)[0]
    return float(alpha), float(beta), n

def car_for_deal(returns, mkt_arr, calendar, announce_date):
    """CAR(EVT_START, EVT_END) from a market model fitted on the estimation window.

    returns: Series of the acquirer's daily returns indexed by date. Returns a dict with
    alpha, beta, n_est, n_event, car and status. A CAR needs at least MIN_EST_OBS
    estimation-window returns and every event-window return.
    """
    out = dict(alpha=np.nan, beta=np.nan, n_est=0, n_event=0, car=np.nan, status="ok")
    i0 = event_day_position(calendar, announce_date)
    if i0 is None or i0 + EST_START < 0 or i0 + EVT_END >= len(calendar):
        out["status"] = "outside_calendar"
        return out
    est = slice(i0 + EST_START, i0 + EST_END + 1)
    evt = slice(i0 + EVT_START, i0 + EVT_END + 1)
    y_est = returns.reindex(calendar[est]).to_numpy(float)
    y_evt = returns.reindex(calendar[evt]).to_numpy(float)
    alpha, beta, n_est = fit_market_model(y_est, mkt_arr[est])
    out.update(alpha=alpha, beta=beta, n_est=n_est, n_event=int((~np.isnan(y_evt)).sum()))
    if n_est < MIN_EST_OBS:
        out["status"] = "few_est_obs"
    elif out["n_event"] < (EVT_END - EVT_START + 1):
        out["status"] = "missing_event_returns"
    else:
        out["car"] = float(np.sum(y_evt - (alpha + beta * mkt_arr[evt])))
    return out

def compute_all_cars(acq, mkt):
    """One row per deal (KEYS) with its market-model CAR and a status code."""
    calendar, mkt_arr = mkt.index, mkt.to_numpy(float)
    rows = []
    for key, g in acq.groupby(KEYS, sort=False):
        res = car_for_deal(g.set_index("dlycaldt")["dlyret"], mkt_arr, calendar, key[2])
        rows.append(
            dict(zip(KEYS, key))
            | dict(permno=g["permno"].iloc[0], anchor_decile=g["anchor_decile"].iloc[0])
            | res
        )
    return pd.DataFrame(rows)

def bhar_for_deal(returns, mkt_arr, calendar, announce_date):
    """Quarterly (q1..q4) and 12-month BHAR for one deal.

    BHAR = acquirer compounded return minus market compounded return over the same days.
    Needs all HORIZON acquirer returns after t = 0; otherwise the BHARs stay NaN and
    bhar_status says why.
    """
    out = dict.fromkeys(["q1", "q2", "q3", "q4", "bhar12"], np.nan)
    out["bhar_status"] = "ok"
    i0 = event_day_position(calendar, announce_date)
    if i0 is None or i0 + HORIZON >= len(calendar):
        out["bhar_status"] = "outside_calendar"
        return out
    window = slice(i0 + 1, i0 + HORIZON + 1)
    y = returns.reindex(calendar[window]).to_numpy(float)
    m = mkt_arr[window]
    if np.isnan(y).any():
        out["bhar_status"] = "incomplete_window"
        return out

    def bh(first, last):
        # compounded acquirer return minus compounded market return over days first..last (1-based)
        return np.prod(1 + y[first - 1:last]) - np.prod(1 + m[first - 1:last])

    for q, (a, b) in enumerate(QUARTERS, 1):
        out[f"q{q}"] = bh(a, b)
    out["bhar12"] = bh(1, HORIZON)
    return out

def compute_all_bhar(acq, mkt):
    """One row per deal (KEYS) with quarterly and 12-month BHAR."""
    calendar, mkt_arr = mkt.index, mkt.to_numpy(float)
    rows = []
    for key, g in acq.groupby(KEYS, sort=False):
        res = bhar_for_deal(g.set_index("dlycaldt")["dlyret"], mkt_arr, calendar, key[2])
        rows.append(dict(zip(KEYS, key)) | res)
    return pd.DataFrame(rows)

def build_panel(cars, bhar):
    """One row per deal: decile, CAR(-5,+5), quarterly BHARs and 12-month BHAR (statuses kept)."""
    c = cars[KEYS + ["anchor_decile", "car", "status"]].rename(columns={"status": "car_status"})
    return c.merge(bhar, on=KEYS, how="left", validate="one_to_one").rename(columns={"anchor_decile": "decile"})

In [8]:
mkt = load_market(FF_DAILY)
acq = attach_deciles(read_acquirer_returns(), sdc)
print(f"{len(mkt):,} trading days ({mkt.index.min().date()} to {mkt.index.max().date()}); {len(acq):,} acquirer-day rows")

15,918 trading days (1963-07-01 to 2026-09-30); 800,529 acquirer-day rows


In [9]:
cars = compute_all_cars(acq, mkt)
bhar = compute_all_bhar(acq, mkt)
panel = build_panel(cars, bhar)
print("CAR status:", cars["status"].value_counts().to_dict())
print("BHAR status:", bhar["bhar_status"].value_counts().to_dict())
print(f"deals: {len(panel)} | with CAR: {panel['car'].notna().sum()} | with full-year BHAR: {panel['bhar12'].notna().sum()}")

CAR status: {'ok': 1441, 'few_est_obs': 17, 'missing_event_returns': 3}
BHAR status: {'ok': 1432, 'incomplete_window': 29}
deals: 1461 | with CAR: 1441 | with full-year BHAR: 1432


## 4. Tests and report tables
Per-decile tests are one-sample t-tests of mean = 0. Spread tests use a Welch t-test, a permutation p-value, a bootstrap interval, and median and winsorized differences; Holm-adjusted p-values cover the four quarterly D1 - D10 tests.
Deals are treated as independent and the per-decile p-values are not adjusted for the number of tests.

In [10]:
def stars(p):
    """Significance stars: *** p<0.01, ** p<0.05, * p<0.10."""
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

def winsorize(s, p=0.01):
    """Clip a Series at its p and 1-p quantiles."""
    return s.clip(s.quantile(p), s.quantile(1 - p))

def decile_stats(panel, periods=PERIODS):
    """Per decile and period: n, mean, median, sd, winsorized mean, t-stat and two-sided p-value for mean = 0."""
    rows = []
    for col, label in periods.items():
        d = panel.dropna(subset=[col])
        w = winsorize(d[col])
        for dec, s in d.groupby("decile")[col]:
            t = stats.ttest_1samp(s, 0.0)
            rows.append(dict(period=label, decile=int(dec), n=len(s), mean=s.mean(), median=s.median(),
                             sd=s.std(), wins_mean=w[s.index].mean(), t=t.statistic, p=t.pvalue,
                             sig=stars(t.pvalue)))
    return pd.DataFrame(rows)

def spread_test(panel, col, label, lo, hi, rng, n_perm=N_PERM, n_boot=N_BOOT):
    """Spread (decile group lo minus group hi): mean/median/winsorized differences,
    Welch t and p, permutation p (shuffling group labels), and a bootstrap 95% interval of the mean difference.
    """
    d = panel.dropna(subset=[col])
    x = d.loc[d.decile.isin(lo), col].to_numpy()
    y = d.loc[d.decile.isin(hi), col].to_numpy()
    diff = x.mean() - y.mean()
    welch = stats.ttest_ind(x, y, equal_var=False)
    pool, cnt = np.concatenate([x, y]), 0
    for _ in range(n_perm):
        rng.shuffle(pool)
        cnt += abs(pool[:len(x)].mean() - pool[len(x):].mean()) >= abs(diff)
    boot = [rng.choice(x, len(x)).mean() - rng.choice(y, len(y)).mean() for _ in range(n_boot)]
    w = winsorize(d[col])
    return dict(period=label, n_lo=len(x), n_hi=len(y), mean_diff=diff, median_diff=np.median(x) - np.median(y),
                wins_mean_diff=w[d.decile.isin(lo)].mean() - w[d.decile.isin(hi)].mean(),
                welch_t=welch.statistic, welch_p=welch.pvalue, perm_p=(cnt + 1) / (n_perm + 1),
                boot_lo=np.percentile(boot, 2.5), boot_hi=np.percentile(boot, 97.5), sig=stars(welch.pvalue))

def holm(p):
    """Holm step-down adjusted p-values for an array of p-values."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - rank) * p[i]))
        adj[i] = run
    return adj

def all_spreads(panel, periods=PERIODS, seed=SEED, n_perm=N_PERM, n_boot=N_BOOT):
    """Spread tests for D1 - D10 and D1-2 minus D9-10 in every period.

    Adds Holm-adjusted p-values across the four quarterly D1 - D10 BHAR tests.
    """
    rng = np.random.default_rng(seed)
    out = []
    for name, lo, hi in [("D1 - D10", [1], [10]), ("D1-2 minus D9-10", [1, 2], [9, 10])]:
        r = pd.DataFrame([spread_test(panel, col, label, lo, hi, rng, n_perm, n_boot)
                          for col, label in periods.items()])
        r.insert(0, "spread", name)
        r["holm_p_quarters"] = np.nan
        if name == "D1 - D10":
            q = r["period"].isin(["BHAR Q1", "BHAR Q2", "BHAR Q3", "BHAR Q4"])
            r.loc[q, "holm_p_quarters"] = holm(r.loc[q, "welch_p"])
        out.append(r)
    return pd.concat(out, ignore_index=True)

def sample_table(panel):
    """Deals per decile: total, with CAR, with a full 12-month BHAR window."""
    g = panel.groupby("decile")
    return pd.DataFrame({"deals": g.size(), "with_car": g["car"].count(), "with_bhar": g["bhar12"].count()})

def wide_report(dec, periods=PERIODS):
    """Report layout: deciles in rows, periods in columns, cells 'mean% (t)' with stars."""
    dec = dec.assign(cell=[f"{m*100:.2f}%{s} ({t:.2f})" for m, t, s in zip(dec["mean"], dec["t"], dec["sig"])])
    return dec.pivot(index="decile", columns="period", values="cell")[list(periods.values())]

def median_table(dec, periods=PERIODS):
    """Median abnormal return by decile (percent), deciles in rows, periods in columns."""
    return (dec.pivot(index="decile", columns="period", values="median")[list(periods.values())] * 100).round(2)

In [11]:
dec = decile_stats(panel)
tables = {
    "sample": sample_table(panel),
    "by_decile": dec,
    "by_decile_wide": wide_report(dec),
    "median": median_table(dec),
    "spreads": all_spreads(panel),
}
print("TABLE 1. Sample by decile\n", tables["sample"].to_string(), "\n")
print("TABLE 2. Mean abnormal return by decile, % (t-stat for mean = 0); * p<0.10, ** p<0.05, *** p<0.01\n",
      tables["by_decile_wide"].to_string(), "\n")
print("TABLE 3. Median abnormal return by decile, %\n", tables["median"].to_string(), "\n")
print("TABLE 4. Spread tests (differences in decimal returns)\n", tables["spreads"].round(4).to_string(index=False))

TABLE 1. Sample by decile
         deals  with_car  with_bhar
decile                            
1         147       144        142
2         146       146        141
3         146       141        141
4         146       142        144
5         146       145        141
6         146       144        144
7         146       145        144
8         146       144        145
9         146       145        144
10        146       145        146 

TABLE 2. Mean abnormal return by decile, % (t-stat for mean = 0); * p<0.10, ** p<0.05, *** p<0.01
 period         CAR[-5,+5]            BHAR Q1           BHAR Q2           BHAR Q3         BHAR Q4           BHAR 12m
decile                                                                                                             
1          -0.48% (-0.43)     -2.68% (-1.28)   -3.48%* (-1.76)      2.57% (0.87)  5.90%** (1.99)     -2.31% (-0.47)
2          -1.19% (-1.35)     -0.54% (-0.37)    -0.45% (-0.28)  -4.07%** (-2.54)  -1.60% (-1.13)   -7.58

## 5. Save outputs
Aggregate tables go to `results/` (tracked in git). The deal-level panel goes to `data/processed/` (not tracked: it is derived from licensed data).

In [12]:
RESULTS.mkdir(exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
panel.to_csv(PROCESSED / "car_bhar_by_deal.csv", index=False)
tables["by_decile"].to_csv(RESULTS / "report_table_by_decile.csv", index=False)
tables["spreads"].to_csv(RESULTS / "report_table_spreads.csv", index=False)
tables["sample"].to_csv(RESULTS / "report_table_sample.csv")
tables["by_decile_wide"].to_csv(RESULTS / "report_table_by_decile_wide.csv")